# Introduction to NLP with Python - pattern matching
## Negation detection

A brief introduction to using Python for negation detection for information extraction using a simple version of the NegEx algorithm.


Written by Sumithra Velupillai May 2020


## The NegEx algorithm

The NegEx algorithm is a widely used algorithm in clinical NLP. It is a simple pattern matching algorithm that relies on two main lexicons:

* a list of terms/concepts that are the main concepts of interest for the information extraction problem, e.g. diagnoses, symptoms. These are called target terms.

* a list of terms that indicate negation. In the original version of NegEx the negation terms were classified as pre-negation terms, post-negation terms, and pseudonegation terms (i.e. terms that are ambiguous).

In simple terms, the algorithm works in the following way:

* For each sentence, look for target terms.
* If a target term is found, check if this term is negated. This is done by looking at the surrounding words in a window of +/- 5 words within the sentence.


The original article:

Chapman et al. A Simple Algorithm for Identifying Negated Findings and Diseases in Discharge Summaries,
Journal of Biomedical Informatics Volume 34, Issue 5, October 2001, Pages 301-310

https://www.sciencedirect.com/science/article/pii/S1532046401910299

There are a few extended versions of this algorithm, where other modifiers are taken into account (e.g. uncertainty, experiencer), where several types of targets can be defined, where the scope of a modifier is dealt with differently, etc.  

Some relevant publications:


Harkema et al. ConText: An Algorithm for Determining Negation, Experiencer, and Temporal Status From Clinical Reports.
J Biomed Inform. 2009 Oct;42(5):839-51. doi: 10.1016/j.jbi.2009.05.002. Epub 2009 May 10.

https://pubmed.ncbi.nlm.nih.gov/19435614/


Chapman et al. Document-Level Classification of CT Pulmonary Angiography Reports based on an Extension of the ConText Algorithm.
J Biomed Inform. 2011 Oct; 44(5): 728–737. doi: 10.1016/j.jbi.2011.03.011
https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3164892/


Chapman et al. Extending the NegEx Lexicon for Multiple Languages
Stud Health Technol Inform. 2013;192:677-81.
https://pubmed.ncbi.nlm.nih.gov/23920642/


Example of using this in the mental health domain:

Downs et al. Detection of Suicidality in Adolescents with Autism Spectrum Disorders: Developing a Natural Language Processing Approach for Use in Electronic Health Records
AMIA Annu Symp Proc. 2017; 2017: 641–649.

https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5977628/


Velupillai et al. Identifying Suicidal Adolescents From Mental Health Records Using Natural Language Processing
Stud Health Technol Inform. 2019 Aug 21;264:413-417. doi: 10.3233/SHTI190254.

https://pubmed.ncbi.nlm.nih.gov/31437956/


We'll use pandas to save outputs

In [1]:
import pandas as pd

A key package for working with pattern matching and regular expressions is called 're', we need to import that too.

In [2]:
import re

We will use SpaCy again for tokenizing.

spaCy: https://spacy.io/


spaCy has a default language model for English that we will load into the variable 'nlp'

In [3]:
try:
    import spacy
except ImportError as e:
    !pip install spacy
    import spacy
try:
    nlp = spacy.load('en_core_web_sm')
except Error as e:
    !python -m spacy download en_core_web_sm
    nlp = spacy.load('en_core_web_sm')


Let's define a function to extract words from sentences, and exclude punctuations

In [4]:
import string
def get_spacy_tokens(row):
  return [str(token) for token in row if str(token) not in string.punctuation]

Then we need a function to implement the NegEx algorithm, that returns a dataframe with each sentence, a list of target terms (if found) and if the sentence is negated or not (boolean)

In [5]:
def simple_negex(doc, target_terms, negation_terms):
    negated_sentences = []
    for sentence in doc.sents:
        words = get_spacy_tokens(sentence)
        negated = False
        ## find target terms
        t_word = []
        neg_word = []
        for w in words:
            negated = False
            for reg in target_terms:
                r = re.compile(reg, flags=re.I)
                if re.search(r, w):
                    # target term found, save
                    t_word.append(w)
                    # look for negation in window +- 5 words
                    start = words.index(w)-6
                    if start<0:
                        start=0
                    for ww in words[start:words.index(w)]:
                        if ww in negation_terms:
                            negated = True
                            break
                    end = words.index(w)+6
                    if end > len(words):
                        end = len(words)
                    for ww in words[words.index(w):end+1]:
                        if ww in negation_terms:
                            negated = True
                            break
            neg_word.append(negated)
        if True in neg_word:
            negated_sentences.append([str(sentence), t_word, True])
        else:
            negated_sentences.append([str(sentence), t_word, False])
    df = pd.DataFrame(negated_sentences, columns=['sentence', 'target terms', 'negated'])
    return df


Let's create a sample document, a list of target terms, and a list of negation terms

In [6]:

text = "The patient denies having suicidal thoughts. This was not an intentional overdose. She has been suicidal in the past. Suicidal ideation was not intentional."

## we'll use spacy for tokenizing
doc = nlp(text)

## a simple list of target terms
target_terms = ['suicid']

## a simple list of negation terms
negation_terms = ['no', 'not']

negated_sentences = simple_negex(doc, target_terms, negation_terms)

What results did we get?

In [7]:
negated_sentences

,sentence,target terms,negated
0,The patient denies having suicidal thoughts.,[suicidal],False
1,This was not an intentional overdose.,[],False
2,She has been suicidal in the past.,[suicidal],False
3,Suicidal ideation was not intentional.,[Suicidal],True


What do you think about these results? Are there any terms missing as targets? As negations?

*Try adding new terms, changing sentences!*

In [7]:
target_terms = [
    r'suicid',          # suicidal, suicide, suicidality
    r'ideation',
    r'self-harm',
    r'self harm',
    r'overdose',
    r'SI\b'             # 如果文本里会用缩写 SI
]

In [8]:
negation_terms = [
    'no', 'not', 'denies', 'denied', 'without',
    'never', 'neither', 'none', 'negative'
]

In [9]:
text = """
The patient denies suicidal ideation.
No evidence of self-harm.
She has a history of suicide attempt.
The patient is not suicidal today.
He presented after intentional overdose.
"""

target_terms = [r'suicid', r'ideation', r'self-harm', r'self harm', r'overdose', r'attempt']
negation_terms = ['no', 'not', 'denies', 'denied', 'without', 'never', 'negative']

In [10]:
# ============================================
# Target-level NegEx-style detection
# More detailed version
# ============================================

import pandas as pd
import re
import string

try:
    import spacy
except ImportError:
    !pip install spacy
    import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except Exception:
    !python -m spacy download en_core_web_sm
    nlp = spacy.load("en_core_web_sm")


def get_spacy_tokens(span):
    return [str(token) for token in span if str(token) not in string.punctuation]


def simple_negex_target_level(doc, target_terms, negation_terms, window_size=5):
    """
    Returns a target-level dataframe:
    one row per target found
    """
    negation_terms_lower = [w.lower() for w in negation_terms]
    rows = []

    for sentence in doc.sents:
        words = get_spacy_tokens(sentence)
        words_lower = [w.lower() for w in words]

        for i, w in enumerate(words):
            for reg in target_terms:
                pattern = re.compile(reg, flags=re.I)
                if re.search(pattern, w):
                    start = max(0, i - window_size)
                    end = min(len(words), i + window_size + 1)

                    left_context = words_lower[start:i]
                    right_context = words_lower[i+1:end]

                    local_negations = [neg for neg in negation_terms_lower
                                       if neg in left_context or neg in right_context]

                    negated = len(local_negations) > 0

                    rows.append({
                        "sentence": str(sentence),
                        "target_term_found": w,
                        "negated": negated,
                        "negation_triggers_found": list(set(local_negations)),
                        "left_context": left_context,
                        "right_context": right_context
                    })

    df = pd.DataFrame(rows)
    return df


# Example text
text = """
The patient denies having suicidal thoughts.
This was not an intentional overdose.
She has been suicidal in the past.
Suicidal ideation was not intentional.
No evidence of self-harm was found.
The patient never expressed suicide intent.
He denied overdose but reported past suicidal ideation.
"""

doc = nlp(text)

target_terms = [
    r"suicid",
    r"ideation",
    r"self-harm",
    r"self harm",
    r"overdose",
    r"attempt",
    r"intent"
]

negation_terms = [
    "no",
    "not",
    "denies",
    "denied",
    "without",
    "never",
    "none",
    "negative"
]

target_level_results = simple_negex_target_level(
    doc=doc,
    target_terms=target_terms,
    negation_terms=negation_terms,
    window_size=5
)

print(target_level_results)

target_level_results.to_csv("negex_target_level_output.csv", index=False)
print("\nSaved results to negex_target_level_output.csv")

                                             sentence target_term_found  \
0    \nThe patient denies having suicidal thoughts.\n          suicidal   
1             This was not an intentional overdose.\n       intentional   
2             This was not an intentional overdose.\n          overdose   
3                She has been suicidal in the past.\n          suicidal   
4            Suicidal ideation was not intentional.\n          Suicidal   
5            Suicidal ideation was not intentional.\n          ideation   
6            Suicidal ideation was not intentional.\n       intentional   
7       The patient never expressed suicide intent.\n           suicide   
8       The patient never expressed suicide intent.\n            intent   
9   He denied overdose but reported past suicidal ...          overdose   
10  He denied overdose but reported past suicidal ...          suicidal   
11  He denied overdose but reported past suicidal ...          ideation   

    negated negation_tri